# Notebook 09: Filter Decapping Candidates

Apply filtering criteria to identify decapping P-body candidate genes.

**Reverse-engineered thresholds (confirmed):**
- `Cap_index <= 0.1` (max in candidates = 0.0998)
- `Cytosol_TPM >= 10` (filename "10threshold", min = 10.34)
- `Pub_Log2FC > 0` (all candidates positive; min = 2.02)

**Expected result:** 374 candidate genes matching Jason's list.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
REFERENCE_DIR = Path("../reference")

In [2]:
merged = pd.read_csv(PROCESSED_DIR / "08_hub_cage_depmap_impc.csv")
print(f"Loaded {len(merged):,} genes")

# Also load reference candidates for comparison
ref_candidates = pd.read_csv(REFERENCE_DIR / "Decapping PB candidates 10threshold+DepMap.csv")
print(f"Reference candidates: {len(ref_candidates):,}")

Loaded 12,811 genes
Reference candidates: 375


## 1. Apply Confirmed Thresholds

Start with the confirmed thresholds and check gene count progressively.

In [3]:
# Find the Pub_Log2FC column
pub_col = None
for c in merged.columns:
    if 'pub' in c.lower() and 'log2' in c.lower():
        pub_col = c
        break
    if c == 'Pub_Log2FC':
        pub_col = c
        break

# Also check for the original column name from mmc3
if pub_col is None:
    for c in merged.columns:
        if 'RNA enrichment' in c and 'log2' in c:
            pub_col = c
            break

print(f"Using Pub_Log2FC column: {pub_col}")

# Progressive filtering
print(f"\n{'Filter':<40} {'Remaining':>10}")
print("-" * 55)
print(f"{'All genes':<40} {len(merged):>10,}")

f1 = merged[merged["Cap_index"] <= 0.1]
print(f"{'Cap_index <= 0.1':<40} {len(f1):>10,}")

f2 = f1[f1["Cytosol_TPM"] >= 10]
print(f"{'+ Cytosol_TPM >= 10':<40} {len(f2):>10,}")

if pub_col:
    f3 = f2[f2[pub_col] > 0]
    print(f"{'+ Pub_Log2FC > 0':<40} {len(f3):>10,}")
else:
    f3 = f2

print(f"\nTarget: 374 candidates")
print(f"Result: {len(f3):,} candidates")

Using Pub_Log2FC column: RNA enrichment in P-body (log2) (Fold change=sorted P-bodies/pre-sorted fraction)

Filter                                    Remaining
-------------------------------------------------------
All genes                                    12,811
Cap_index <= 0.1                              3,018
+ Cytosol_TPM >= 10                             885
+ Pub_Log2FC > 0                                698

Target: 374 candidates
Result: 698 candidates


In [4]:
# If count doesn't match 374, try additional thresholds
candidates = f3.copy()

if len(candidates) != 374:
    print(f"Count mismatch: got {len(candidates)}, expected 374")
    print(f"\nTrying additional thresholds...")
    
    # Test Pub_Log2FC threshold variations
    if pub_col:
        for thresh in [0, 0.5, 1.0, 1.5, 2.0]:
            n = len(f2[f2[pub_col] > thresh])
            marker = " <-- MATCH" if n == 374 else ""
            print(f"  Pub_Log2FC > {thresh}: {n:,}{marker}")
    
    # Test with FDR threshold
    fdr_col = [c for c in merged.columns if 'fdr' in c.lower() or 'adjusted p' in c.lower()]
    if fdr_col:
        for thresh in [0.01, 0.05, 0.1]:
            base = f2[f2[pub_col] > 0] if pub_col else f2
            n = len(base[base[fdr_col[0]] < thresh])
            marker = " <-- MATCH" if n == 374 else ""
            print(f"  + FDR < {thresh}: {n:,}{marker}")
    
    # Test with NEW_Log2FC_TPM threshold
    if "NEW_Log2FC_TPM" in merged.columns:
        for thresh in [0, 0.5, 1.0]:
            base = f2[f2[pub_col] > 0] if pub_col else f2
            n = len(base[base["NEW_Log2FC_TPM"] > thresh])
            marker = " <-- MATCH" if n == 374 else ""
            print(f"  + NEW_Log2FC_TPM > {thresh}: {n:,}{marker}")
else:
    print("Count matches! Proceeding with these thresholds.")

Count mismatch: got 698, expected 374

Trying additional thresholds...
  Pub_Log2FC > 0: 698
  Pub_Log2FC > 0.5: 639
  Pub_Log2FC > 1.0: 588
  Pub_Log2FC > 1.5: 507
  Pub_Log2FC > 2.0: 375
  + FDR < 0.01: 626
  + FDR < 0.05: 638
  + FDR < 0.1: 650
  + NEW_Log2FC_TPM > 0: 604
  + NEW_Log2FC_TPM > 0.5: 538
  + NEW_Log2FC_TPM > 1.0: 430


## 2. Gene-for-Gene Comparison

In [5]:
# Compare our candidates against Jason's reference list
# Reference uses Associated Gene Name (no ensembl_id column)
gene_name_col = None
for c in candidates.columns:
    if 'associated gene name' in c.lower() or c == 'Associated Gene Name':
        gene_name_col = c
        break

if gene_name_col:
    our_genes = set(candidates[gene_name_col].astype(str))
    ref_genes = set(ref_candidates["Associated Gene Name"].astype(str))
    
    in_both = our_genes & ref_genes
    in_ours_only = our_genes - ref_genes
    in_ref_only = ref_genes - our_genes
    
    print(f"Our candidates: {len(our_genes):,}")
    print(f"Reference candidates: {len(ref_genes):,}")
    print(f"In both: {len(in_both):,}")
    print(f"In ours only: {len(in_ours_only)}")
    print(f"In reference only: {len(in_ref_only)}")
    
    if in_ours_only:
        print(f"\nGenes in our list but NOT in reference:")
        for g in sorted(in_ours_only):
            row = candidates[candidates[gene_name_col] == g].iloc[0]
            print(f"  {g}: Cap_index={row['Cap_index']:.6f}, Cytosol_TPM={row['Cytosol_TPM']:.2f}")
    
    if in_ref_only:
        print(f"\nGenes in reference but NOT in our list:")
        for g in sorted(in_ref_only):
            row = ref_candidates[ref_candidates["Associated Gene Name"] == g].iloc[0]
            print(f"  {g}: Cap_index={row['Cap_index']:.6f}, Cytosol_TPM={row['Cytosol_TPM']:.2f}")
else:
    print("Could not find gene name column for comparison")

Our candidates: 698
Reference candidates: 375
In both: 358
In ours only: 340
In reference only: 17

Genes in our list but NOT in reference:
  ABHD10: Cap_index=0.095852, Cytosol_TPM=200.12
  ABL1: Cap_index=0.060136, Cytosol_TPM=75.87
  ACER2: Cap_index=0.010175, Cytosol_TPM=20.38
  ACRV1: Cap_index=0.000000, Cytosol_TPM=10.91
  ACSS3: Cap_index=0.000000, Cytosol_TPM=10.66
  ACVR1B: Cap_index=0.086013, Cytosol_TPM=72.33
  ADNP2: Cap_index=0.065051, Cytosol_TPM=82.89
  ADRB2: Cap_index=0.064930, Cytosol_TPM=17.57
  AFF3: Cap_index=0.073737, Cytosol_TPM=19.69
  AGK: Cap_index=0.087945, Cytosol_TPM=70.74
  AGTPBP1: Cap_index=0.071057, Cytosol_TPM=161.97
  AKAP5: Cap_index=0.079953, Cytosol_TPM=28.53
  AKAP7: Cap_index=0.090862, Cytosol_TPM=14.84
  AKT3: Cap_index=0.090134, Cytosol_TPM=138.05
  ALDH1L2: Cap_index=0.026111, Cytosol_TPM=11.91
  AMPH: Cap_index=0.095363, Cytosol_TPM=31.53
  ANGPTL1: Cap_index=0.000000, Cytosol_TPM=10.49
  ANKRD18A: Cap_index=0.049664, Cytosol_TPM=14.61
  ANKR

## 3. Save Output

In [6]:
# Save candidates with columns matching Jason's format
output_cols = [
    gene_name_col, "gene_length_bp", "FANTOM_Total_CAGE_TPM",
    "Cytosol_TPM", "PB_TPM"
]
if pub_col:
    output_cols.append(pub_col)
output_cols.extend([
    "NEW_Log2FC_TPM", "Cap_index",
    "DepMap_Common_Essential", "DepMap_Strongly_Selective",
    "DepMap_Flag_Status", "DepMap_Dependent_Cell_Lines"
])

# Filter to existing columns
output_cols = [c for c in output_cols if c in candidates.columns]

candidates_out = candidates[output_cols].copy()
output_path = PROCESSED_DIR / "09_decapping_candidates.csv"
candidates_out.to_csv(output_path, index=False)
print(f"Saved {len(candidates_out):,} decapping candidates to {output_path}")
print(f"Columns: {list(candidates_out.columns)}")

Saved 698 decapping candidates to ../data/processed/09_decapping_candidates.csv
Columns: ['Associated Gene Name', 'gene_length_bp', 'FANTOM_Total_CAGE_TPM', 'Cytosol_TPM', 'PB_TPM', 'RNA enrichment in P-body (log2) (Fold change=sorted P-bodies/pre-sorted fraction)', 'NEW_Log2FC_TPM', 'Cap_index', 'DepMap_Common_Essential', 'DepMap_Strongly_Selective', 'DepMap_Flag_Status', 'DepMap_Dependent_Cell_Lines']
